# OP-01 · Téléchargement et contrôle qualité C3S

**Notebook opérationnel** — à exécuter chaque mois, au début du cycle.

| | |
|---|---|
| Étape du workflow | E2 — acquisition des sorties de modèles |
| Entrée | `config/cycle_YYYYMM.yaml` |
| Sorties | `DATA_OSF/raw/c3s/YYYYMM/*.grib` · `OUTPUTS_OSF/qc/YYYYMM/c3s_<variable>_*.csv` · manifestes dans `OUTPUTS_OSF/runs/` |
| Durée | 3 à 4 h au premier passage (file d'attente du CDS) ; quelques minutes ensuite |

**Mode d'emploi :** renseigner la cellule *Paramètres*, puis *Exécuter tout*. Les fichiers déjà téléchargés ne sont pas retéléchargés. Pour relire les requêtes sans rien télécharger, mettre `DRY_RUN = True`.

Équivalent en ligne de commande :
```
python scripts/run_download_c3s.py --config config/cycle_YYYYMM.yaml --variable precip
python scripts/run_qc_c3s.py       --config config/cycle_YYYYMM.yaml --variable precip
```

## Paramètres

In [ ]:
CYCLE_CONFIG = "config/cycle_202609.yaml"   # fichier du cycle du mois
VARIABLE     = "precip"                     # "precip" (température : phase P1)
MODELS       = None                         # None = tous les modèles de la configuration, ou ["ecmwf", "dwd"]
KINDS        = ["forecast", "hindcast"]
DRY_RUN      = False                        # True : afficher les requêtes sans télécharger

In [ ]:
from pathlib import Path
import os, json
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(REPO)

from eccas_s2s.settings import load_cycle
from eccas_s2s.operations import download_c3s, qc_c3s
from eccas_s2s.core.periods import build_periods, periods_table

cfg = load_cycle(CYCLE_CONFIG)
print(f"Cycle {cfg.cycle_id} — initialisation {cfg.init_date.date()} — {len(cfg.c3s_models)} modèles C3S")
print("Données brutes :", cfg.raw_dir("c3s"))

## 1. Téléchargement

In [ ]:
dl = download_c3s.run(CYCLE_CONFIG, VARIABLE, kinds=KINDS, models=MODELS, dry_run=DRY_RUN)
failures = dl.parameters.get("failures", {})
print(f"\nStatut : {dl.status} — {len(dl.outputs)} fichier(s) — {len(failures)} échec(s)")
if failures:
    display(pd.Series(failures, name="erreur"))

## 2. Contrôle qualité des fichiers

Pour chaque fichier : nombre de membres et d'années, échéances reçues, horizon disponible pour **toutes** les années, échéances manquantes. UKMO et BoM ont peu de membres au 1er du mois (ensembles décalés). Ils sont conservés tels quels (décision du 19/09/2026) et signalés ici.

In [ ]:
if DRY_RUN:
    print("DRY_RUN = True : contrôle qualité non exécuté.")
else:
    qc, horizons, qc_ctx = qc_c3s.run(CYCLE_CONFIG, VARIABLE)
    cols = ["centre", "system", "kind", "members", "years", "n_steps", "horizon_all_years_days",
            "horizon_any_year_days", "size_mb", "warnings"]
    display(qc[cols])

## 3. Horizon utilisable et cohérence avec la configuration

In [ ]:
if not DRY_RUN:
    h = horizons.copy()
    h["max_lead_days (config)"] = h["centre"].map({c: m.max_lead_days for c, m in cfg.c3s_models.items()})
    h["cohérent"] = h["usable_days"] == h["max_lead_days (config)"]
    display(h)
    if not h["cohérent"].all():
        print("⚠ Mettre à jour max_lead_days dans", CYCLE_CONFIG, "pour les modèles non cohérents.")
    if qc_ctx.warnings:
        print("\nAlertes du contrôle qualité :")
        for w in qc_ctx.warnings:
            print(" -", w)

## 4. Périodes produites avec l'horizon commun

In [ ]:
if not DRY_RUN:
    common = int(horizons["usable_days"].min())
    tab = periods_table(build_periods(cfg.init_date, common, cfg.scales), cfg.init_date.year)
    print(f"Horizon commun : {common} jours → jusqu'au {(cfg.init_date + pd.Timedelta(days=common - 1)).date()}")
    display(tab.groupby("scale")["label"].agg(["count", "first", "last"]))

## 5. Traçabilité

In [ ]:
for c in [dl] + ([] if DRY_RUN else [qc_ctx]):
    print(f"{c.step:24s} {c.status:8s} {c.run_dir / 'manifest.json'}")